# Fase 1 — Máscaras de referência (GEE)

Construção das máscaras de referência candidatas para a classe **café** na área de estudo (Região Geográfica Imediata de Guaxupé - MG).

Este estágio prepara duas fontes: a classificação **MapBiomas** (classe café = 46) binarizada a 10 m e um agrupamento não supervisionado (**k-means**) sobre os embeddings anuais **AlphaEarth**. Ambas são exportadas como GeoTIFF para o Google Drive e servirão de base à comparação de fontes de verdade de campo na Fase 2.

## Detecção da raiz do repositório

Localiza a raiz do repositório a partir do diretório corrente e a insere no caminho de importação, garantindo o acesso ao pacote `src/`.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path


# Sobe os diretórios até encontrar src/config.yaml, marcador da raiz do projeto.
def _find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "src" / "config.yaml").is_file():
            return candidate
    raise RuntimeError("Raiz do repositório não localizada (src/config.yaml ausente).")


PROJECT_ROOT = _find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f"Raiz do projeto: {PROJECT_ROOT}")

## Detecção da plataforma

Identifica o ambiente de execução (Kaggle, Colab ou local) para adaptar a instalação de dependências e a leitura de segredos.

In [ ]:
import importlib.util
import os


# Heurística por variáveis de ambiente e presença de diretórios característicos.
def detect_platform() -> str:
    if "KAGGLE_KERNEL_RUN_TYPE" in os.environ or Path("/kaggle").is_dir():
        return "kaggle"
    if "COLAB_GPU" in os.environ or importlib.util.find_spec("google.colab") is not None:
        return "colab"
    return "local"


PLATFORM = detect_platform()
print(f"Plataforma detectada: {PLATFORM}")

## Instalação condicional das dependências

Em Kaggle/Colab instala o pacote com os extras geoespaciais e de aprendizado de máquina. No ambiente local a instalação é ignorada, pois é gerenciada por `uv` e pelo CI.

In [ ]:
import subprocess

# Instala o projeto editavelmente com os extras necessários apenas em nuvem.
if PLATFORM in {"kaggle", "colab"}:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-e", ".[geo,ml]"],
        cwd=PROJECT_ROOT,
        check=True,
    )
    print("Dependências instaladas.")
else:
    print("Ambiente local: instalação ignorada (gerenciada por uv/CI).")

## Carregamento da configuração única

Lê a configuração de `src/config.yaml` por meio de `src/config.py`, fonte única de verdade dos assets de máscara, do ano de referência e dos parâmetros do k-means.

In [ ]:
from src.config import CONFIG

# Exibe os parâmetros das máscaras de referência.
print(f"Ano de referência: {CONFIG.get('masks.year')} | Classe café: {CONFIG.get('masks.coffee_class')}")
print(f"MapBiomas 10 m: {CONFIG.get('masks.mapbiomas_10m_asset')}")
print(f"AlphaEarth: {CONFIG.get('masks.alphaearth_collection')} | clusters: {CONFIG.get('masks.alphaearth_clusters')}")

## Fixação das sementes

Fixa as sementes de `python`, `numpy`, `torch` e `cuda`; a semente também alimenta a amostragem do k-means, tornando a máscara candidata reprodutível.

In [ ]:
from src.config import seed_everything

# Aplica a semente global definida na configuração.
resolved_seed = seed_everything()
print(f"Sementes fixadas em {resolved_seed}.")

## Carregamento de segredos

Em Kaggle/Colab injeta os segredos do cofre da plataforma nas variáveis de ambiente esperadas pelo pacote. Nenhum valor é impresso. No ambiente local, os segredos devem vir de variáveis de ambiente ou do arquivo `.env`.

In [ ]:
# Nomes das variáveis de ambiente consumidas pela aquisição.
SECRET_NAMES = (
    "GEE_SERVICE_ACCOUNT_EMAIL",
    "GEE_PROJECT",
    "GEE_SERVICE_ACCOUNT_KEY_JSON",
)

if PLATFORM == "kaggle":
    from kaggle_secrets import UserSecretsClient

    client = UserSecretsClient()
    for name in SECRET_NAMES:
        try:
            os.environ[name] = client.get_secret(name)
        except Exception:
            print(f"Segredo ausente no Kaggle: {name}")
elif PLATFORM == "colab":
    from google.colab import userdata

    for name in SECRET_NAMES:
        try:
            os.environ[name] = userdata.get(name)
        except Exception:
            print(f"Segredo ausente no Colab: {name}")
else:
    print("Ambiente local: segredos esperados via variáveis de ambiente/.env.")

## Autenticação no Google Earth Engine

Inicializa o Earth Engine com a conta de serviço lida exclusivamente do ambiente e interrompe a execução caso as credenciais estejam ausentes.

In [ ]:
from src.data.gee_client import init_ee

# Inicializa o cliente do Earth Engine; sem credenciais a fase não prossegue.
ee = init_ee()
print("Earth Engine autenticado com sucesso.")

## Carregamento da área de estudo

Carrega a malha vetorial do IBGE, recorta a Região Geográfica Imediata de Guaxupé e converte a geometria para o formato do Earth Engine.

In [ ]:
from src.data.aoi import geometry_to_ee, get_region_geometry

# Obtém a geometria unificada da região e a converte para ee.Geometry.
aoi_geometry = get_region_geometry()
aoi_ee = geometry_to_ee(ee, aoi_geometry)
print(f"Área de estudo: {CONFIG.get('aoi.region_name')} ({CONFIG.get('aoi.region_code')})")

## Construção da máscara MapBiomas

Seleciona a classificação MapBiomas do ano de referência e a binariza, atribuindo 1 aos pixels da classe café (46) e 0 aos demais.

In [ ]:
from src.data.mask_utils import build_mapbiomas_coffee_mask

# Gera a máscara binária de café a partir da classificação MapBiomas.
mapbiomas_mask = build_mapbiomas_coffee_mask(ee, aoi_ee)
print(f"Banda da máscara MapBiomas: {mapbiomas_mask.bandNames().getInfo()}")

## Verificação da distribuição de classes

Calcula o histograma de frequência da máscara MapBiomas na área de estudo para confirmar a presença de pixels de café antes da exportação.

In [ ]:
# Conta pixels por valor (0 e 1) dentro da área de estudo.
histogram = mapbiomas_mask.reduceRegion(
    reducer=ee.Reducer.frequencyHistogram(),
    geometry=aoi_ee,
    scale=CONFIG.get("gee.scale_m"),
    maxPixels=CONFIG.get("gee.max_pixels"),
    bestEffort=True,
).getInfo()
print(f"Histograma da máscara MapBiomas: {histogram}")

## Mosaico dos embeddings AlphaEarth

Monta o mosaico anual dos embeddings AlphaEarth (64 bandas) recortado pela área de estudo, usado como representação não supervisionada do uso do solo.

In [ ]:
from src.data.mask_utils import get_alphaearth_embedding

# Mosaico anual dos embeddings AlphaEarth para o ano de referência.
embedding = get_alphaearth_embedding(ee, aoi_ee)
print(f"Número de bandas do embedding: {embedding.bandNames().size().getInfo()}")

## Clusterização k-means dos embeddings

Treina um clusterizador k-means sobre amostras dos embeddings e atribui um rótulo de cluster a cada pixel, gerando a máscara candidata não supervisionada.

In [ ]:
from src.data.mask_utils import cluster_alphaearth

# Agrupa os embeddings com semente fixa para reprodutibilidade.
clusters = cluster_alphaearth(ee, embedding, aoi_ee)
print(f"Banda de clusters: {clusters.bandNames().getInfo()}")

## Exportação das máscaras candidatas

Dispara as exportações GeoTIFF da máscara MapBiomas e do mapa de clusters AlphaEarth para o Google Drive, com descrições determinísticas.

In [ ]:
from src.data.gee_client import export_image_to_drive, make_export_description

# Exporta a máscara MapBiomas binária de café.
mapbiomas_description = make_export_description(
    f"{CONFIG.get('gee.export_prefix')}_mapbiomas_coffee",
    CONFIG.get("aoi.region_code"),
    CONFIG.get("gee.start_date"),
    CONFIG.get("gee.end_date"),
)
mapbiomas_task = export_image_to_drive(
    ee,
    mapbiomas_mask,
    description=mapbiomas_description,
    region=aoi_ee,
    subfolder=CONFIG.get("masks.export_subfolder"),
    file_name_prefix=mapbiomas_description,
)
print(f"Tarefa MapBiomas: {mapbiomas_description} | id={mapbiomas_task.id}")

## Exportação do mapa de clusters AlphaEarth

Dispara a exportação GeoTIFF do mapa de clusters AlphaEarth, que será comparado às demais fontes na Fase 2.

In [ ]:
# Exporta o mapa de clusters AlphaEarth como máscara candidata.
alphaearth_description = make_export_description(
    f"{CONFIG.get('gee.export_prefix')}_alphaearth_clusters",
    CONFIG.get("aoi.region_code"),
    CONFIG.get("gee.start_date"),
    CONFIG.get("gee.end_date"),
)
alphaearth_task = export_image_to_drive(
    ee,
    clusters,
    description=alphaearth_description,
    region=aoi_ee,
    subfolder=CONFIG.get("masks.export_subfolder"),
    file_name_prefix=alphaearth_description,
)
print(f"Tarefa AlphaEarth: {alphaearth_description} | id={alphaearth_task.id}")

## Acompanhamento das tarefas de exportação

Lista o estado das tarefas do Earth Engine para confirmar a conclusão das exportações das máscaras candidatas.

In [ ]:
import time

# Consulta o estado das tarefas recentes até todas concluírem.
while True:
    statuses = [t.status() for t in ee.batch.Task.list()[:5]]
    states = [s.get("state") for s in statuses]
    print(f"Estados: {states}")
    if all(state in {"COMPLETED", "FAILED", "CANCELLED"} for state in states):
        break
    time.sleep(30)